In [ ]:
pip install -U langchain langchain-community accelerate bitsandbytes pypdf sentence-transformers chromadb langchain_google_vertexai langchain-huggingface

In [ ]:
import pandas as pd
import time
import datetime

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.prompts import SystemMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.document_transformers import LongContextReorder
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain.retrievers import ContextualCompressionRetriever, MultiQueryRetriever

from langchain_community.vectorstores import Chroma, FAISS
from langchain_google_vertexai import ChatVertexAI, embeddings as VertexAIEmbeddings, HarmBlockThreshold, HarmCategory
from langchain_community.chat_models import ChatCohere, ChatOllama
from langchain_community.embeddings import CohereEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
def PrepareLLMAndChainNoRAG_Cohere(api_key):
    chatLlm = ChatCohere(cohere_api_key=api_key, temperature=0.5)

    # Define the chat prompt template
    systemMessage = SystemMessagePromptTemplate.from_template("""
    ¿Que nota entre 0 y 10 tendría el alumno a la pregunta de examen {question}?\n                                       
    Responda solamente con la nota redondeada a 2 decimales. No se requiere justificación.
    """)

    chat_template = ChatPromptTemplate.from_messages(
        [
            systemMessage,
            ("human", "La respuesta del alumno es: {input}")
        ]
    )

    # Create the retrieval chain
    return chat_template | chatLlm | StrOutputParser()

In [ ]:
def PrepareLLMAndRetrieval_Cohere(api_key:str, documentsPath: str, isASingleFile: bool, retrieverType: str="SingleQuery"):
    chatLlm = ChatCohere(cohere_api_key=api_key, temperature=0.5)

    # Define the chat prompt template
    systemMessage = SystemMessagePromptTemplate.from_template("""
    Atendiendo solamente al contexto proporcionado, ¿Que nota entre 0 y 10 tendría el alumno a la pregunta de examen {question}?\n
    El contexto proporcionado es el siguiente:
    <context>{context}</context>
                                                            
    Responda solamente con la nota redondeada a 2 decimales. No se requiere justificación.
    """)

    chat_template = ChatPromptTemplate.from_messages(
        [
            systemMessage,
            ("human", "La respuesta del alumno es: {input}")
        ]
    )

    # Load the document or all the documents in the directory
    # and split them into chunks
    if isASingleFile:
        loader = PyPDFLoader(documentsPath)
    else:
        loader = DirectoryLoader(documentsPath, glob="**/*.pdf", loader_cls=PyPDFLoader)
    docs =  loader.load_and_split()
   
    # Create the vector store
    embeddings = CohereEmbeddings(cohere_api_key=api_key)
    vector = FAISS.from_documents(docs, embeddings)

    # Create the retrieval chain
    if retrieverType == "SingleQuery":
        retriever = vector.as_retriever()
    elif retrieverType == "MultiQuery":
        retriever = MultiQueryRetriever.from_llm(retriever=vector.as_retriever(), llm=chatLlm)
    elif retrieverType == "ContextualCompressionRetriever":
        filterPrompt = PromptTemplate.from_template("¿Is the context <context>{context}</context> relevant to the question \"{question}\"?. Response Yes or No")
        _filter = LLMChainFilter.from_llm(chatLlm,filterPrompt)
        retriever = ContextualCompressionRetriever(base_compressor=_filter, base_retriever=vector.as_retriever())
    
    return (retriever, chat_template | chatLlm | StrOutputParser())

In [ ]:
# Uses LLama3 from Ollama https://ollama.com/
def PrepareLLMAndChainNoRAG_Llama3():
    chatLlm = ChatOllama(model="llama3", stop=["<|eot_id|>"], temperature=0.5)

    # Define the chat prompt template
    systemMessage = SystemMessagePromptTemplate.from_template("""
    ¿Que nota entre 0 y 10 tendría el alumno a la pregunta de examen {question}?\n                                       
    Responda solamente con la nota redondeada a 2 decimales. No se requiere justificación.
    """)

    chat_template = ChatPromptTemplate.from_messages(
        [
            systemMessage,
            ("human", "La respuesta del alumno es: {input}")
        ]
    )

    return chat_template | chatLlm | StrOutputParser()

In [ ]:
# Uses LLama3 from Ollama https://ollama.com/
def PrepareLLMAndRetrieval_Llama3(documentsPath: str, isASingleFile: bool, retrieverType: str="SingleQuery"):
    Llama3EmbeddingsModelId = "sentence-transformers/all-mpnet-base-v2"
    Llama3Embeddings = HuggingFaceEmbeddings(model_name=Llama3EmbeddingsModelId)    
    chatLlm = ChatOllama(model="llama3", stop=["<|eot_id|>"], temperature=0.5)

    # Define the chat prompt template
    systemTemplate = """
    Eres un profesor que tiene que corregir preguntas de exámenes.
    Responda con la frase "La puntuación es: " seguida de la puntuación obtenida por el alumno entre 0 y 10 redondeada a 2 decimales.
    La puntuación es 0 cuando la respuesta es totalmente incorrecta y 10 cuando es perfecta.
    La pregunta a corregir es: 
    "{question}"
    Para poner la puntuación debes tener solamente en cuenta el siguiente courserio:
    {context}
    """

    template = f"""
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    "{systemTemplate}"
    <|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    Responda con la frase "La puntuación es: " seguida de la puntuación obtenida por el alumno entre 0 y 10 redondeada a 2 decimales.
    La respuesta del alumno que tienes que corregir es: "{{input}}"
    <|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """

    # Added prompt template
    prompt = PromptTemplate(
        input_variables=["question", "input"],
        template=template
    )

    # Load the documents
    if isASingleFile:
        loader = PyPDFLoader(documentsPath)
    else:
        loader = DirectoryLoader(documentsPath, glob="**/*.pdf", loader_cls=PyPDFLoader)
    docs =  loader.load_and_split()
   
    # Create the vector store
    vector = Chroma.from_documents(documents=docs, embedding=Llama3Embeddings, persist_directory="chroma_db_llama3")
    
    # Create the retrieval chain
    if retrieverType == "SingleQuery":
        retriever = vector.as_retriever()
    elif retrieverType == "MultiQuery":
        retriever = MultiQueryRetriever.from_llm(retriever=vector.as_retriever(), llm=chatLlm)
    elif retrieverType == "ContextualCompressionRetriever":
        filterPrompt = PromptTemplate.from_template("¿Is the context <context>{context}</context> relevant to the question \"{question}\"?. Response Yes or No")
        _filter = LLMChainFilter.from_llm(chatLlm,filterPrompt)
        retriever = ContextualCompressionRetriever(base_compressor=_filter, base_retriever=vector.as_retriever())

    return (retriever, prompt | chatLlm | StrOutputParser())

In [ ]:
def PrepareLLMAndChainNoRAG_Gemini(api_key: str):

    chatLlm = ChatVertexAI(model="gemini-pro", google_api_key=api_key)

    # Define the chat prompt template
    systemMessage = SystemMessagePromptTemplate.from_template("""
    ¿Que nota entre 0 y 10 tendría el alumno a la pregunta de examen {question}?\n                                       
    Responda solamente con la nota redondeada a 2 decimales. No se requiere justificación.
    """)

    chat_template = ChatPromptTemplate.from_messages(
        [
            systemMessage,
            ("human", "La respuesta del alumno es: {input}")
        ]
    )

    # Create the retrieval chain
    return chat_template | chatLlm | StrOutputParser()

In [ ]:
def PrepareLLMAndRetrieval_Gemini(api_key: str, documentsPath: str, isASingleFile: bool, retrieverType: str="SingleQuery"):
    safeSettings = {HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE
                    , HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE
                    , HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE
                    , HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE
                    , HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE
                    }
    chatLlm = ChatVertexAI(model="gemini-1.5-pro", google_api_key=api_key, temperature=0.5
                           , safety_settings=safeSettings)

    # Define the chat prompt template
    systemMessage = SystemMessagePromptTemplate.from_template("""
    Atendiendo solamente al contexto proporcionado, ¿Que nota entre 0 y 10 tendría el alumno a la pregunta de examen {question}?\n
    El contexto proporcionado es el siguiente:
    <context>{context}</context>
                                                            
    Responda solamente con la nota redondeada a 2 decimales. No se requiere justificación.
    """)

    chat_template = ChatPromptTemplate.from_messages(
        [
            systemMessage,
            ("human", "La respuesta del alumno es: {input}")
        ]
    )

    # Load the documents
    if isASingleFile:
        loader = PyPDFLoader(documentsPath)
    else:
        loader = DirectoryLoader(documentsPath, glob="**/*.pdf", loader_cls=PyPDFLoader)
    docs =  loader.load_and_split()

    # Create the vector store
    embeddings = VertexAIEmbeddings.VertexAIEmbeddings(model_name="text-embedding-004")
    vector = Chroma.from_documents(docs, embeddings)

    # Create the retrieval chain
    if retrieverType == "SingleQuery":
        retriever = vector.as_retriever()
    elif retrieverType == "MultiQuery":
        retriever = MultiQueryRetriever.from_llm(retriever=vector.as_retriever(), llm=chatLlm)
    elif retrieverType == "ContextualCompressionRetriever":
        filterPrompt = PromptTemplate.from_template("¿Is the context <context>{context}</context> relevant to the question \"{question}\"?. Response Yes or No")
        _filter = LLMChainFilter.from_llm(chatLlm,filterPrompt)
        retriever = ContextualCompressionRetriever(
            base_compressor=_filter, base_retriever=vector.as_retriever()
        )

    return (retriever, chat_template | chatLlm | StrOutputParser())

In [ ]:
def ReadExamAnswers(genericPath, questionsFile, studentAnswersFile):
    questions = open(genericPath + "\\" + questionsFile, 'r', encoding="utf-8")
    lines = questions.readlines()

    question = ""
    answer = ""
    isquestion = True
    for line in lines:
        if line.strip() == "":
            continue

        if isquestion:
            question += line.strip() + " "
        else:
            answer += line.strip() + " "
        
        isquestion = not isquestion
    
    studentAnswers = pd.read_csv(genericPath + "\\" + studentAnswersFile,
                        sep=",", encoding="utf-8")
    
    #adding question to the student answers
    studentAnswers["Pregunta"] = question.strip()

    #adding perfect answer to the student answers
    perfectAnswer = {'Pregunta': question, 
                     'RespuestaEstudiante': answer, 
                     'NotaProfesorSemántica(sobre 0.75)': 1,
                     'NotaFinal (sobre 10)': 10} 
    
    studentAnswers.loc[len(studentAnswers)] = perfectAnswer
    studentAnswers["LlmGrade"] = ""
    return studentAnswers

In [ ]:
def getResultsFileName(model, isASingleFile, retrieverType, course, resultsBasePath: str):
    if retrieverType == "NoRAG":
        return resultsBasePath + "\\" + model + "\\" + f"results{course}_" + retrieverType + ".csv"
    else:
        if isASingleFile:
            return resultsBasePath + "\\" + model + "\\" + f"results{course}_singleCourserio_" + retrieverType + ".csv"
        else:
            return resultsBasePath + "\\" + model + "\\" + f"results{course}_multipleCourserio_" + retrieverType + ".csv"

In [ ]:
def EvaluateStudentAnswers(retrieval_chain, studentAnswers, modelName, retriever: None):
    context = None
    for index, row in studentAnswers.iterrows():
        print(f'Question nº {index}')
        question = row["Pregunta"]
        studentAnswer = row["RespuestaEstudiante"]

        # distinguish between using RAG or not
        if retriever is None:
            # get response (numerical grade) from the LLM
            response = retrieval_chain.invoke({"question": question, 
                                        "input": studentAnswer})
        else:
            # At this point, it is being evaluated only one question, so retrieve the context only once and use it for all the answers
            # It is done to avoid the cost of retrieving the context for each answer and also to be consistent with the context used for all the answers
            if context is None:
                print(f'Retrieving context start {datetime.datetime.now()}')  
                contextResults = retriever.invoke(input = question)
                print(f'Retrieving context end {datetime.datetime.now()}')
                reordering = LongContextReorder()
                reordered_docs = reordering.transform_documents(contextResults)
                distinctContextResults = list(set(doc.page_content for doc in reordered_docs))
                distinctContextResults.reverse()
                context="\n".join(distinctContextResults)
            
            # get response (numerical grade) from the LLM
            response = retrieval_chain.invoke({"context": context,
                                        "question": question, 
                                        "input": studentAnswer})            
        
        # update the studentAnswers dataframe with the response (numerical grade)
        studentAnswers.iloc[index, studentAnswers.columns.get_loc('LlmGrade')] = response

        # because using the free version of LLMs, wait to avoid the limit of requests per minute
        if modelName == "Gemini":
            time.sleep(60)
        elif modelName == "Cohere":
            time.sleep(20)

        if index >= 1:
            print(response)
            break 
    return studentAnswers

In [ ]:
# Used to get the grade for all the courses students responses
def getResultsForLLmAndRetriever(api_key, model, isASingleFile, retrieverType, courses, courseMaterialsPath, studentAnswersPath, answerFilePrefix, resultsBasePath: str):
    finalResults = None
    for courseId in courses:
        course = "Course" + str(courseId)
        print(f"Processing {course}... Model: {model} SingleFile: {isASingleFile} retrieverType: {retrieverType}")

        # document paths
        documentPath = courseMaterialsPath
        if isASingleFile:
            documentPath = f"{documentPath}\\{course}.pdf"

        # get student answers
        studentAnswers = ReadExamAnswers(studentAnswersPath , f"PreguntasTema{courseId}.txt", f"{answerFilePrefix}Tema{courseId}.csv")

        # get retrieval & chain
        chain = None
        retriever = None
        if retrieverType == "NoRAG":
            if model == "Cohere":
                chain = PrepareLLMAndChainNoRAG_Cohere(api_key)
            elif model == "Llama3":
                chain = PrepareLLMAndChainNoRAG_Llama3()
            if model == "Gemini":
                chain = PrepareLLMAndChainNoRAG_Gemini(api_key)
        else:
            if model == "Cohere":
                retriever, chain = PrepareLLMAndRetrieval_Cohere(api_key, documentPath, isASingleFile, retrieverType)
            elif model == "Llama3":
                retriever, chain = PrepareLLMAndRetrieval_Llama3(documentPath, isASingleFile, retrieverType)
            if model == "Gemini":
                retriever, chain = PrepareLLMAndRetrieval_Gemini(api_key, documentPath, isASingleFile, retrieverType)

        # evaluate student answers
        studentAnswers = EvaluateStudentAnswers(chain, studentAnswers, model, retriever)
        # save results
        studentAnswers.to_csv(getResultsFileName(model, isASingleFile, retrieverType, course, resultsBasePath), index=False, encoding="utf-8")

        # concatenate results to have all in one file
        if finalResults is None:
            finalResults = studentAnswers
        else:
            finalResults = pd.concat([finalResults, studentAnswers], axis=0)

    # save all results in one file
    finalResults.to_csv(getResultsFileName(model, isASingleFile, retrieverType, "all", resultsBasePath), index=False, encoding="utf-8")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [ ]:
def main(api_key, model, courses, courseMaterialsPath, studentAnswersPath, answerFilePrefix, resultsBasePath):
    if not os.path.exists(resultsBasePath):
        os.mkdir(resultsBasePath)

    isASingleFile = True    # True, False
    retrieverType = ""      # NoRAG, SingleQuery, MultiQuery, ContextualCompressionRetriever
    
    for i in range(4):
        if i == 0:
            isASingleFile = True
            retrieverType = "NoRAG"
        elif i == 1:
            isASingleFile = True
            retrieverType = "SingleQuery"
        elif i == 2:
            isASingleFile = False
            retrieverType = "SingleQuery"
        elif i == 3:
            isASingleFile = True
            retrieverType = "ContextualCompressionRetriever"
        elif i == 4:
            isASingleFile = True
            retrieverType = "MultiQuery"

        getResultsForLLmAndRetriever(api_key, model, isASingleFile, retrieverType, courses, courseMaterialsPath, studentAnswersPath, answerFilePrefix, resultsBasePath)

In [ ]:

model = "Gemini"         # Cohere, Llama3, Gemini
api_key = "your_api_key" # Cohere API key, Ollama API key, Gemini API
courseMaterialsPath = "path_to_course_materials" # Path to the course materials
studentAnswersPath = "path_to_student_answers" # Path to the student answers
answerFilePrefix = "answer_file_prefix" # Prefix for the answer files, all the answer files should be in the same folder and have the same prefix. Ex: Subject1Tema1.csv, Subject1Tema2.csv, etc.
courses = [] # List of courses to process, if empty all the courses will be processed
resultsBasePath = "path_to_results" # Path to save the results

main(api_key, model, courses, courseMaterialsPath, studentAnswersPath, answerFilePrefix, resultsBasePath)